# 05 — RAG Chat Logic (Ollama)

Where retrieval meets reasoning: take a question, pull the most relevant facts from Notebook 4's index, hand them to a local LLM with strict instructions to use *only* that context, and return an answer with its sources — "The Voice" from the original pitch.

**Running on Ollama** — free, fully local, no API key or billing account needed. The trade-off: local models small enough to run on a laptop reason noticeably less carefully than a hosted frontier model, so the prompt below is written to be extra explicit about structure and honesty, which helps smaller models perform closer to their ceiling.

## Before running this: make sure Ollama itself is installed and running

The `ollama` **Python package** below only talks to a local Ollama **server** — it doesn't include the server itself. If you haven't already:

1. Install Ollama from [ollama.com/download](https://ollama.com/download) (separate from anything pip installs)
2. Pull a model once, from a terminal: `ollama pull llama3.2`
3. Ollama needs to actually be running in the background for any of the cells below to work — the desktop app keeps it running automatically; if you installed it another way, run `ollama serve` in a terminal first

In [1]:
!pip install ollama -q


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Reload the index and facts

Notebook 4 already built and saved the FAISS index — no need to rebuild it. Note: `DATA_DIR` here is `"data"`, not `"../data"` like Notebooks 1-4, since this one's being run with the working directory set to the project root rather than from inside `notebooks/`. Worth picking one convention and sticking to it across all five notebooks so it's not confusing for Iliya or Mani — adjust this path to match wherever you actually run it from.

In [2]:
import pandas as pd
import faiss
import ollama
from sentence_transformers import SentenceTransformer

DATA_DIR = "data"
facts_df = pd.read_csv(f"{DATA_DIR}/facts.csv")
index = faiss.read_index(f"{DATA_DIR}/facts.faiss")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Loaded {len(facts_df)} facts and an index of {index.ntotal} vectors")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded 709 facts and an index of 709 vectors


## Retrieval

Bumped `k` from 5 to 7 — with a small model doing the reasoning, it helps to hand it slightly more candidate facts to choose from, since it's less able to work well with a narrow, imperfect set.

In [3]:
MIN_SCORE = 0.35

def search(query, k=7, min_score=MIN_SCORE):
    q_vec = embed_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_vec)
    scores, ids = index.search(q_vec, k)
    return [
        {"text": facts_df.iloc[i]["text"], "category": facts_df.iloc[i]["category"],
         "n": int(facts_df.iloc[i]["n"]), "score": float(s)}
        for i, s in zip(ids[0], scores[0]) if s >= min_score
    ]

## The prompt

Tightened compared to the first draft. Smaller local models tend to hedge everything and dump raw numbers in a pile rather than writing naturally — so this version is explicit about *format*, not just content: lead with a direct answer, weave sample sizes into sentences instead of listing them, and name specifically what's missing rather than talking around it.

In [4]:
SYSTEM_PROMPT = (
    "You are a career advisor for software developers, answering strictly from the survey "
    "facts provided below. Rules:\n"
    "1. Use ONLY these facts — never add outside knowledge or general opinions about "
    "languages, tools, or careers.\n"
    "2. Start with a direct, clear answer in 1-2 sentences, then support it with specific facts.\n"
    "3. Weave each sample size naturally into its sentence (e.g. 'based on 200 respondents') "
    "— never list several sample sizes together as a bare group of numbers.\n"
    "4. If part of the question isn't covered by the facts (for example, a specific technology "
    "with no matching fact), say plainly which part is missing rather than talking around it.\n"
    "5. Keep the whole answer under 150 words."
)

def build_user_message(question, retrieved_facts):
    context = "\n".join(f"- {f['text']}" for f in retrieved_facts)
    return f"Context (2025 Stack Overflow Developer Survey):\n{context}\n\nQuestion: {question}"

## Tying it together

In [5]:
OLLAMA_MODEL = "llama3.2"

def ask(question, k=7):
    retrieved = search(question, k=k)
    if not retrieved:
        return {
            "answer": "I don't have survey data that speaks to that directly — I can only answer from the 2025 Stack Overflow Developer Survey facts I've indexed.",
            "sources": [],
        }
    user_message = build_user_message(question, retrieved)
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
    )
    return {"answer": response["message"]["content"], "sources": retrieved}


def show(question):
    result = ask(question)
    print(f"Q: {question}\n")
    print(f"A: {result['answer']}\n")
    if result["sources"]:
        print("Sources:")
        for s in result["sources"]:
            print(f"  [{s['category']}, n={s['n']}] {s['text'][:80]}")
    print("\n" + "="*80 + "\n")

## Test it

Same three questions as before — but this time Rust and Go actually have coverage: Notebook 3 now generates a dedicated salary fact for each of the top 20 languages, independent of role, so this should come back with real numbers instead of the closest-available Python stats it was forced to lean on last time.

In [6]:
show("I'm a junior dev in India learning Python. Should I learn Rust or Go next for the best salary bump?")

Q: I'm a junior dev in India learning Python. Should I learn Rust or Go next for the best salary bump?

A: Based on the survey data, it appears that learning Go could be a viable option for you to increase your salary. The survey does not provide direct information about salaries for Go developers, but it shows that Go is used by 14% of respondents in India (based on 514 respondents), which is lower than Python's 57%. Additionally, there are no explicit comparisons or figures for Rust vs. Go salaries in the provided data.

Sources:
  [salary_by_language, n=3036] Developers who use Rust report a median annual salary of \$85,534, based on 3036
  [salary_by_language, n=12417] Developers who use Python report a median annual salary of \$80,000, based on 12
  [language_by_country, n=2020] In India, 57% of developers used Python in the past year, based on 2020 responde
  [tech_by_role, n=809] Among DevOps engineer or professional respondents, 77% used Python as their lang
  [tech_by_role, n=

In [7]:
#show("Is a CS degree worth it compared to being self-taught?")
#show("What is the best pizza topping?")

## What changed from the first draft

- Switched from a hosted API to Ollama — free, local, no billing account needed
- `k` raised from 5 to 7, giving the smaller model more candidate facts to draw on
- Prompt rewritten to be explicit about *structure*, not just rules — smaller models follow format instructions more reliably than open-ended ones like "weigh larger samples more heavily"
- Removed the unused API-key cell entirely rather than leaving it commented out

## The gap this surfaced — now fixed

The Rust-vs-Go test question — straight from the original pitch — didn't have supporting facts the first time this ran: Notebook 3's per-role tech facts only captured the *top 3* languages per role, and Rust/Go never made that cut anywhere. Notebook 3 now has a dedicated salary-and-adoption-by-language section covering the top 20 languages directly, independent of role, which closes that gap — re-run Notebook 3 and 4 if you haven't already, so this notebook is searching the updated 709-fact index.

**Next:** port `ask()` into `app.py`, a standalone Streamlit script — same as planned, just calling Ollama instead of a hosted API now.